# JSON Inverted Index Experiment

Compare article-level BM25 retrieval with and without the legal dictionary. Each JSON index maps a term to article IDs and term frequencies.

In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd()
if ROOT.name == 'evaluation':
    ROOT = ROOT.parents[1]
elif not (ROOT / 'apps').exists():
    ROOT = Path('/Users/nak/CADT/projects/legal-rag')
sys.path.insert(0, str(ROOT))
from experiments.evaluation.json_inverted_index_experiment import OUTPUT, search, tokenize_variant, load_dictionary

def load_index(name):
    path = OUTPUT / f'{name}_inverted_index.json'
    return json.loads(path.read_text(encoding='utf-8'))

without_dictionary = load_index('without_dictionary')
with_dictionary = load_index('with_dictionary')
print('Without dictionary:', without_dictionary['document_count'], 'documents,', without_dictionary['vocabulary_size'], 'terms')
print('With dictionary:   ', with_dictionary['document_count'], 'documents,', with_dictionary['vocabulary_size'], 'terms')

Without dictionary: 5111 documents, 8682 terms
With dictionary:    5111 documents, 8913 terms


## One test query
Change only `query` to test another legal question. Results show article IDs and BM25 scores.

In [4]:
query = "ស្រវឹងបើកបរ"
for name, index in [('without_dictionary', without_dictionary), ('with_dictionary', with_dictionary)]:
    print(f'\n{name}:')
    for rank, (article_id, score) in enumerate(search(index, query, top_k=10), start=1):
        metadata = index['documents'][article_id]
        print(f'{rank:>2}. {article_id:<24} score={score:.4f} | {metadata["title"]}')


without_dictionary:
 1. 4044-62                  score=17.6780 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
 2. 148-63                   score=17.5596 | ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2015)
 3. 4044-82                  score=15.3838 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
 4. 4044-81                  score=15.2306 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
 5. 4044-80                  score=14.9344 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
 6. 4044-65                  score=14.4737 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
 7. 148-8                    score=14.4236 | ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2015)
 8. 4044-9                   score=14.3784 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
 9. 148-43                   score=14.3142 | ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2015)
10. 1-8                      score=14.1490 | ច្បាប់ស្តីពីចរាចរផ្លូវគោក (វិសោធនកម្ម)

with_dictionary:
 1. 4044-62  

## Inspect term → documents
This shows the postings for one normalized term in both indexes. `df` is the number of articles containing the term.

In [3]:
term_query = 'ស្រវឹងបើកបរ'
for name, index in [('without_dictionary', without_dictionary), ('with_dictionary', with_dictionary)]:
    dictionary = load_dictionary() if index['dictionary_enabled'] else None
    terms = tokenize_variant(term_query, dictionary)
    print(f'\n{name}; query tokens: {terms}')
    for term in terms:
        posting = index['terms'].get(term, {'df': 0, 'postings': {}})
        print(f'\nTERM={term!r} DF={posting["df"]}')
        for article_id, frequency in list(posting['postings'].items())[:20]:
            print(f'  {article_id}: tf={frequency} | {index["documents"][article_id]["title"]}')


without_dictionary; query tokens: ['ស្រវឹង', 'បើក', 'បរ']

TERM='ស្រវឹង' DF=24
  1-8: tf=1 | ច្បាប់ស្តីពីចរាចរផ្លូវគោក (វិសោធនកម្ម)
  1-77: tf=1 | ច្បាប់ស្តីពីចរាចរផ្លូវគោក (វិសោធនកម្ម)
  1231-115: tf=2 | ច្បាប់ស្តីពីការងារ
  148-8: tf=1 | ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2015)
  148-43: tf=2 | ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2015)
  148-63: tf=2 | ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2015)
  148-66: tf=1 | ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2015)
  148-76: tf=1 | ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2015)
  148-82: tf=1 | ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2015)
  148-84: tf=1 | ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2015)
  4044-9: tf=1 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
  4044-43: tf=1 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
  4044-62: tf=2 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
  4044-65: tf=1 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
  4044-66-2: tf=1 | ច្បាប់ស្តីពី វិសោធនកម្ម ច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក (2007)
  4044-76: tf=1 | ច្បាប់ស្